# Writer Palmyra Vision on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Writer's Palmyra Vision 7B reads images and answers questions about them. It accepts tool
definitions but never calls a tool, and rejects every tool choice except `none`, so this
notebook has no tool-use section. It refuses the `system` role as well, and every turn
after the first has to alternate, so instructions go in the first user turn (section 4). Writer's text models, Palmyra X4 and X5, are on
`bedrock-runtime` only and have their own notebook,
[`02-palmyra-x4-x5-on-runtime`](02-palmyra-x4-x5-on-runtime.ipynb).

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Context | Max output | Notes |
|---|---|---|---|---|---|
| [Palmyra Vision 7B](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-writer-palmyra-vision-7b.html) | `writer.palmyra-vision-7b` | `writer.palmyra-vision-7b` | 4,096 tokens, shared with the output | 4,096, less what the input uses | Vision and language |

IDs, context window and output limit as of October 2026, in `us-east-1`. Which Regions
carry each model changes as models launch, so the setup cell checks your Region.

On `bedrock-mantle` these models serve the OpenAI-compatible **Chat Completions** API at
the bare `/v1` path. On `bedrock-runtime` it serves **Converse** under the same ID
(section 6).

The 4,096-token window covers the prompt and the answer together, so an image leaves less
room for `max_tokens` than a question does, and a budget that fills the window is refused
before the model runs (section 7).

**Covered:** first call, vision, streaming, multi-turn, structured output, Converse on
`bedrock-runtime`, and sizing a request against the 4,096-token window.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes an availability
check and a lenient JSON parser, plus the sample images and their known answers for the
vision cells.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for, parse_json_lenient

REGION = "us-east-1"
VISION = "writer.palmyra-vision-7b"

# bedrock-mantle: the OpenAI SDK, with a short-term Bedrock API key minted from your
# AWS credentials. Mint it when you build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

# bedrock-runtime: boto3 signs each request with SigV4, so there is no key to mint.
runtime = boto3.client("bedrock-runtime", region_name=REGION)

for model in (VISION,):
    served = [name for name, ok in endpoints_for(model, REGION).items() if ok]
    print(f"{model:26} served by: {', '.join(served) or 'neither endpoint in ' + REGION}")

writer.palmyra-vision-7b   served by: mantle, runtime


## 1. First call

A single request with a short question. If the model spends the whole `max_tokens`
budget before it answers, `content` is `None` and `finish_reason` is `length`, so the
cell prints both.

In [2]:
response = client.chat.completions.create(
    model=VISION,
    messages=[{"role": "user", "content": "What is the capital of Australia? Answer in one word."}],
    max_tokens=1000,
)
choice = response.choices[0]
print((choice.message.content or "").strip())
print(f"finish_reason={choice.finish_reason}  total_tokens={response.usage.total_tokens}")

Canberra
finish_reason=stop  total_tokens=18


## 2. Vision

Send the image as a data URL in an `image_url` content part, next to the text part. The
slide comes from a public AWS talk and ships with this repository, so the answer is
known: its title is `Ingestion from database`. Ask Palmyra Vision to read text off an
image, and check what it reads. Asked to describe this slide, it invented parts that are
not on it in every September 2026 run (Elasticsearch, "data destruction", a datacenter).
Asked for the title only, it read it exactly in most runs, and sometimes changed or
added a word ("Ingestion from databases", "Ingestion from database to database"). The
cell compares the whole answer with the title, so a misreading shows as a mismatch
instead of passing a substring check.

In [3]:
from bedrock import SLIDE_TITLE, slide_data_url

response = client.chat.completions.create(
    model=VISION,
    messages=[{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": slide_data_url()}},
        {"type": "text", "text": "What is this slide's title? Answer with the title only."},
    ]}],
    max_tokens=60,
)
text = response.choices[0].message.content or ""
reading = " ".join(text.lower().replace('"', "").strip().rstrip(".").split())
print(text.strip())
print(f"\nmatches the title exactly: {reading == SLIDE_TITLE.lower()}")

Ingestion from database

matches the title exactly: True


## 3. Streaming

Set `stream=True` and read `delta.content` from each chunk. Check that a chunk has
choices before indexing it, and read `finish_reason` from the last one: `stop` means the
answer is complete.

In [4]:
stream = client.chat.completions.create(
    model=VISION,
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas."}],
    max_tokens=1000,
    stream=True,
)
finish_reason = None
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
    if chunk.choices and chunk.choices[0].finish_reason:
        finish_reason = chunk.choices[0].finish_reason
print(f"\nfinish_reason={finish_reason}")

 1, 2, 3, 4, 5, 6, 7, 8, 9, 10


finish_reason=stop


## 4. Multi-turn

Chat Completions keeps no conversation state on the server, so each request carries the
whole history. Palmyra Vision refuses the `system` role with a 400
`Conversation roles must alternate user/assistant/user/assistant/...`, and sends the same
400 for two user turns in a row (October 2026). So instructions go in the first user turn,
and each reply you keep goes back as an `assistant` turn, even where the next question
does not depend on it.

In [5]:
messages = [
    {"role": "user", "content": "My favourite colour is teal. Reply with one word: noted."},
]
first = client.chat.completions.create(model=VISION, messages=messages, max_tokens=1000)
messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "What is my favourite colour? One word."})
second = client.chat.completions.create(model=VISION, messages=messages, max_tokens=1000)

print("turn 1:", (first.choices[0].message.content or "").strip())
print("turn 2:", (second.choices[0].message.content or "").strip())

turn 1: Noted.
turn 2: Teal.


## 5. Structured output from an image

`response_format` with a JSON Schema works on image requests too. The invoice image
ships with this repository and holds the same four fields as the text extraction in the
other family notebooks, so every value can be checked. Check `finish_reason` first,
because a cut-off answer is cut-off JSON. In September 2026 runs Palmyra Vision misread
a field in about one run in four, nearly always writing the vendor as `ACME`, which is
why the cell compares each value with the image.

In [6]:
from bedrock import INVOICE_FIELDS, invoice_data_url

SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}

response = client.chat.completions.create(
    model=VISION,
    messages=[{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": invoice_data_url()}},
        {"type": "text", "text": (
            "Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the "
            "total as a number without a thousands separator, so 2,345.60 becomes 2345.6.")},
    ]}],
    response_format={"type": "json_schema", "json_schema": {"name": "invoice", "strict": True, "schema": SCHEMA}},
    max_tokens=800,
)
choice = response.choices[0]
if choice.finish_reason != "stop":
    raise RuntimeError(f"incomplete output (finish_reason={choice.finish_reason}); raise max_tokens")

invoice = parse_json_lenient(choice.message.content)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in INVOICE_FIELDS.items() if invoice.get(k) != v}
print("values match the image" if not wrong else f"values that do NOT match the image: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "ACME Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values that do NOT match the image: {'vendor': 'ACME Pty Ltd'}


## 6. On `bedrock-runtime`: Converse

Palmyra Vision is on `bedrock-runtime` under the same ID. Converse has one request shape
for every provider: `content` is a list of blocks, and settings such as the token limit
go in `inferenceConfig`. Read text by block type instead of taking `content[0]`, so the
same code works for models that return other blocks first. The Converse tool loop is in
[`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb).

In [7]:
response = runtime.converse(
    modelId="writer.palmyra-vision-7b",
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 2000},
)
blocks = response["output"]["message"]["content"]
text = "".join(b["text"] for b in blocks if "text" in b)
print("blocks    :", [next(iter(b)) for b in blocks])
print("answer    :", text.strip())
print("stopReason:", response["stopReason"])

blocks    : ['text']
answer    : Canberra
stopReason: end_turn


## 7. Sizing a request against the 4,096-token window

Input and output share one 4,096-token window, so what you can ask for in `max_tokens`
depends on what the prompt already costs. An image costs more than the text beside it: with
one short question, a prompt carrying the sample slide came to 755 prompt tokens, one
carrying the invoice 1,211, one carrying both 1,951, and the question on its own 14 (as of
October 2026). A budget that leaves no room for the input is refused before the model
runs, so measure the prompt first and size the budget from the headroom that is left.
Things that differ quotes the errors from both ends of the window.

Whether the model reads the id correctly varies from run to run, which is section 5's
subject and the *Values read from an image* row's; what this cell checks is the sizing.

In [8]:
WINDOW = 4096   # context window from the model card, shared between input and output
WANT = 500      # the most this answer should need

parts = [
    {"type": "image_url", "image_url": {"url": invoice_data_url()}},
    {"type": "text", "text": "What is the invoice id? Answer with the id only."},
]
# A first call with a tiny budget, only to price the prompt. CountTokens is not supported
# for this model, and guessing is not safe: the request is rejected, rather than
# truncated, when max_tokens leaves no room for the input, and an image's cost is not
# visible from the request.
priced = client.chat.completions.create(
    model=VISION, messages=[{"role": "user", "content": parts}], max_tokens=16,
)
used = priced.usage.prompt_tokens
budget = min(WANT, WINDOW - used)
print(f"prompt_tokens={used}  headroom={WINDOW - used}  max_tokens={budget}")

response = client.chat.completions.create(
    model=VISION,
    messages=[{"role": "user", "content": parts}],
    max_tokens=budget,
)
choice, usage = response.choices[0], response.usage
answer = (choice.message.content or "").strip()
print(f"finish_reason={choice.finish_reason}  prompt={usage.prompt_tokens} "
      f"completion={usage.completion_tokens}  total={usage.total_tokens} of {WINDOW}")
print("answer:", answer)
# What this section claims is that a budget sized from the measured headroom comes back
# complete and inside the window. Both checks can come out false: too small a budget
# returns finish_reason "length", and one that fills the window is refused outright.
print("answer complete          :", choice.finish_reason == "stop")
print("request inside the window:", usage.total_tokens <= WINDOW)


prompt_tokens=1213  headroom=2883  max_tokens=500


finish_reason=stop  prompt=1213 completion=7  total=1220 of 4096
answer: INV-1042
answer complete          : True
request inside the window: True


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Responses API | `POST /v1/responses` returns 400 `The model '…' does not support the '/v1/responses' API` | Use Chat Completions. |
| Path prefix | On `bedrock-mantle`, `/openai/v1/chat/completions` returns 400 ``model `…` isn't supported on this route``. On `bedrock-runtime` that path answers 200 for `writer.palmyra-vision-7b` under its bare ID (October 2026) | Use the bare `/v1` path on `bedrock-mantle`. The prefix is not interchangeable between the endpoints: `bedrock-runtime` has an `/openai/v1` route of its own that serves Palmyra Vision, so the mantle 400 describes that route rather than the models. |
| Conversation state | No `previous_response_id` | Send the full `messages` list on every turn. |
| Empty answer | HTTP 200, `content` is `None`, `finish_reason` is `length` | The model spent `max_tokens` before answering, often on reasoning. Raise the budget within the headroom the prompt leaves (section 7) and check `finish_reason` before using the text. |
| Output budget near the window | A request is refused once `max_tokens` plus the prompt exceeds 4,096, so where that starts depends on the prompt: with the 5-token prompt measured here it is `max_tokens` 4092, which on `bedrock-mantle` returns 400 `validation_error`: `This model's maximum context length is 4096 tokens. However, you requested 4092 output tokens and your prompt contains at least 5 input tokens, for a total of at least 4097 tokens. Please reduce the length of the input prompt or the number of requested output tokens.` Above the window, `max_tokens` 8000 returns 400 `invalid_request_error`: ``'max_tokens' (8000) exceeds model maximum (4096)``. Through Converse, `maxTokens` 4096 raises `ValidationException` wrapping the first message, and 8000 raises `ValidationException: The maximum tokens you requested exceeds the model limit of 4096. Try again with a maximum tokens value that is lower than 4096.` (October 2026) | Size `max_tokens` from the headroom the prompt leaves rather than from the 4,096 limit (section 7). |
| Prompt too long | 400 `validation_error`, counting the input instead: `… you requested 50 output tokens and your prompt contains at least 4047 input tokens, for a total of at least 4097 tokens …` (October 2026) | Images count towards the same window, and cost more than the text beside them: with one short question, a prompt carrying the sample slide came to 755 prompt tokens, one carrying the invoice 1,211, one carrying both 1,951, and the question alone 14. Shorten the prompt, or send one image at a time. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | 429 under load | Throttling is token-based and there is no requests-per-minute quota. Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Metrics | Nothing under `AWS/Bedrock` in CloudWatch | Mantle metrics are in the `AWS/BedrockMantle` namespace. |
| Tool calls | `tools` is accepted and then never used. The default `tool_choice` of `auto` returns 400 `validation_error`: `"auto" tool choice requires --enable-auto-tool-choice and --tool-call-parser to be set`, and `tool_choice="required"` returns `tool_choice="required" requires --tool-call-parser to be set`. Only `tool_choice="none"` answers 200, in prose, with `tool_calls` `None` (October 2026) | There is no tool calling. Leave `tools` out of the request and use structured output (section 5) for machine-readable answers. |
| Values read from an image | A misread field, and not only a cosmetic one, and the rate itself moves between sessions: asking for the invoice id with one prompt returned the right id in 22 of 24 runs in one session and in 6 of 12 in another, both in `us-east-1` on 10 and 11 October 2026. The wrong answers ran from `INV-10442`, the right id with a digit inserted, to strings that are not on the invoice at all, such as `8867071` and `DZ-2340912`. Every one came back with `finish_reason` `stop` | Treat a single read as a sample, not as the value. Check extracted values against the source (section 5), ask for structured output where the shape matters, and read anything you depend on more than once. |
| `system` role | 400 `validation_error`: `Conversation roles must alternate user/assistant/user/assistant/...` | Put instructions in the first user turn (section 4). |
| Turn order | The same 400 for two user turns in a row: `Conversation roles must alternate user/assistant/user/assistant/...` | Append the model's reply as an `assistant` turn before the next user turn, even where you do not need it, so the roles alternate (section 4). |

## Next

- Other vision models: [`04-qwen/02`](../04-qwen/02-qwen3-coder-and-vision.ipynb)
  (Qwen3-VL),
  [`10-nvidia-nemotron/01`](../10-nvidia-nemotron/01-nemotron-nano-and-super.ipynb)
  (Nano 12B v2)
- Writer's text models:
  [`02-palmyra-x4-x5-on-runtime`](02-palmyra-x4-x5-on-runtime.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)